# Agentic Phishing Email Detection and Explanation

Given a raw email, this notebook returns a **verdict** (phishing / suspicious / legitimate),
a **risk score** (0-100), **reasons** backed by evidence, and a **recommended action**.

Three agents work together:
- **Technical Analyst**: deterministic Python checks on headers and URLs (no LLM on a normal run).
- **Content Analyst**: an LLM that reads the text and identifies intent and social-engineering tactics.
- **Verdict and Explainer**: merges both reports. If they disagree, it asks one analyst a single follow-up question.

Run the cells top to bottom. Each stage ends with a **checkpoint** cell that tells you whether the stage worked.

## Stage 0: Config and setup

Everything you might want to change lives in the next cell. No other cell hardcodes model names or limits.

In [ ]:
# ---- Config: the only place to change models, limits and paths ----
import os, sys

# Used for ALL evaluation runs. (The local runner can swap it with --model, for development only.)
# gemini-3.1-flash-lite was the first choice, but it returned "high demand" 503s on most calls in October 2026.
PRIMARY_MODEL  = os.environ.get("PHISH_MODEL", "gemini-3.5-flash-lite")
FALLBACK_MODEL = "gemini-3.1-flash-lite"   # used only if the primary fails the smoke test

RPM_LIMIT       = 15     # requests per minute on the free tier
RPD_LIMIT       = 500    # requests per day (resets at midnight Pacific)
RPD_SAFETY_STOP = 450    # stop evaluation runs here to leave headroom
THROTTLE_RPM    = 12     # our own pace, a little under RPM_LIMIT
MAX_RETRIES     = 8      # retries for rate-limit and server errors, with growing waits (up to ~6 minutes in total)

USE_CACHE_ONLY = False   # True = never call the API; only use cached results (for the demo, or offline)

SEED             = 42
PROMPT_VERSION   = "v3"  # bump this when a prompt changes, so old cache entries are ignored
BODY_CHAR_LIMIT  = 6000  # email bodies are truncated to this before going to the LLM

# Score thresholds that turn the risk score into a label
PHISHING_THRESHOLD   = 70   # score >= 70 -> phishing
SUSPICIOUS_THRESHOLD = 40   # 40-69 -> suspicious, below 40 -> legitimate

IN_COLAB = "google.colab" in sys.modules
# Colab keeps data, cache and results on Google Drive. A local run keeps them in a
# git-ignored "runtime" folder next to the notebook.
DRIVE_ROOT = "/content/drive/MyDrive/phish_project" if IN_COLAB else os.path.abspath("runtime")

CREWAI_VERSION = "1.15.28"

# The Stage 1 smoke test costs about 3 API requests. It runs by default; the local
# runner script turns it off (RUN_SMOKE_TEST=0) so repeated dev runs don't spend quota.
RUN_SMOKE_TEST = os.environ.get("RUN_SMOKE_TEST", "1") == "1"

### Install

CrewAI is pinned to one version, with its `google-genai` extra so it can talk to Gemini directly.
It needs Python 3.10–3.13. If pip says it changed packages that were already loaded,
use **Runtime → Restart session**, then run the cells again from the top. (The install is quick the second time.)

In [ ]:
import subprocess
print("Python", sys.version.split()[0], "| running in", "Colab" if IN_COLAB else "a local Jupyter kernel")
assert (3, 10) <= sys.version_info[:2] < (3, 14), "CrewAI 1.15.x needs Python 3.10-3.13"

# Locally, the packages come from requirements.txt in the project venv instead.
if IN_COLAB:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           f"crewai[google-genai]=={CREWAI_VERSION}", "tldextract"])

In [ ]:
# Turn off CrewAI's anonymous telemetry; nothing about our runs needs to leave the machine except the Gemini calls.
os.environ["CREWAI_TELEMETRY_OPT_OUT"] = "true"
os.environ["OTEL_SDK_DISABLED"] = "true"

import crewai, google.genai, pydantic
print("crewai      ", crewai.__version__)
print("google-genai", google.genai.__version__)
print("pydantic    ", pydantic.__version__)

# Persistent storage for the cache and results, so an interrupted run can resume.
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
for sub in ["data", "cache", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
print("Storage folder:", DRIVE_ROOT)

# The API key never appears in a cell.
# In Colab it comes from Colab Secrets (key icon in the left sidebar).
# Locally it comes from a git-ignored .env file next to the notebook, with one line: GEMINI_API_KEY=...
if IN_COLAB:
    from google.colab import userdata
    _key = userdata.get("GEMINI_API_KEY")
else:
    _key = os.environ.get("GEMINI_API_KEY")
    if not _key and os.path.exists(".env"):
        for line in open(".env", encoding="utf-8"):
            name, _, value = line.strip().partition("=")
            if name.strip() == "GEMINI_API_KEY":
                _key = value.strip().strip('"')
assert _key, "No GEMINI_API_KEY found (Colab Secrets, or .env when running locally)"
os.environ["GEMINI_API_KEY"] = _key   # CrewAI and google-genai both read this name
del _key                              # don't leave it lying around in a variable
print("key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

**✅ Checkpoint 0:** the cell above prints three version numbers, the storage folder, and `key loaded: True`.
It must not print the key itself.

## Stage 1: Smoke test

This is the riskiest part of the project, so it runs before anything else is built. It tests three things:
1. The model names in the config exist on your account.
2. A plain call through CrewAI's Gemini integration works.
3. A CrewAI agent can return **structured output** (a pydantic object), on both the primary and the fallback model.

This stage uses about 3 requests of your daily quota.

In [ ]:
# 1. Do the configured model names exist? Listing models does not use generation quota.
from google import genai

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
available = sorted(m.name.removeprefix("models/") for m in client.models.list())

print("Flash Lite models on this key:")
for name in available:
    if "flash-lite" in name:
        print("  ", name)

for name in [PRIMARY_MODEL, FALLBACK_MODEL]:
    print(f"{name}: {'found' if name in available else 'NOT FOUND, fix the config cell'}")

In [ ]:
# 2. One plain call through CrewAI's native Gemini LLM.
from crewai import LLM

def make_llm(model_name):
    # temperature=0 makes answers as repeatable as the model allows
    return LLM(model=f"gemini/{model_name}", temperature=0)

if RUN_SMOKE_TEST:
    reply = make_llm(PRIMARY_MODEL).call("Reply with the single word OK.")
    print("Reply:", repr(reply))
else:
    print("Smoke test switched off (RUN_SMOKE_TEST is False).")

### The Content Analyst's output schema

The Content Analyst must return this structure instead of free text. Writing it as a pydantic model
means Python checks every field and type for us. The same schema is used in the real system later.

In [ ]:
from typing import Literal
from pydantic import BaseModel, Field

Intent = Literal["credential_theft", "payment_request", "personal_info", "malware_delivery",
                 "advance_fee", "account_verification", "benign", "other"]
TacticName = Literal["urgency", "authority", "fear", "reward", "scarcity", "curiosity"]

class Tactic(BaseModel):
    tactic: TacticName
    quote: str = Field(description="exact words copied from the email that show this tactic")
    explanation: str

class ContentReport(BaseModel):
    intent: Intent
    tactics: list[Tactic]
    content_risk: int = Field(ge=0, le=100, description="0 = clearly harmless, 100 = clearly malicious")
    summary: str

In [ ]:
# 3. A single agent and task that must return a ContentReport.
from crewai import Agent, Task, Crew
from concurrent.futures import ThreadPoolExecutor

def run_crew(crew, inputs):
    """Colab already runs an asyncio event loop, and CrewAI refuses to start a normal
    (synchronous) kickoff inside one. A separate worker thread has no event loop,
    so we run the kickoff there and wait for its result."""
    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(crew.kickoff, inputs=inputs).result()

# Synthetic test email, written for this notebook. It is not a real person's email.
SMOKE_EMAIL = """Subject: Urgent: your account will be suspended

Dear customer,

We detected unusual sign-in activity on your account. To avoid permanent suspension,
verify your identity within 24 hours by logging in here: http://secure-paypa1-login.xyz/verify

Thank you,
Security Team"""

def smoke_test(model_name):
    analyst = Agent(
        role="Content Analyst",
        goal="Identify what an email is trying to make the reader do, and which manipulation tactics it uses.",
        backstory="You study phishing emails and describe them precisely, quoting the email as evidence.",
        llm=make_llm(model_name),
        allow_delegation=False,   # no handing work to other agents
        max_iter=1,               # one attempt, so one task = one request
        verbose=False,
    )
    task = Task(
        description=("Analyse this email. Quote its exact words as evidence for each tactic.\n\n"
                     "EMAIL:\n{email}"),
        expected_output="A ContentReport.",
        agent=analyst,
        output_pydantic=ContentReport,
    )
    result = run_crew(Crew(agents=[analyst], tasks=[task], verbose=False), {"email": SMOKE_EMAIL})

    print(f"=== {model_name} ===")
    print("Token usage:", result.token_usage)
    if result.pydantic is None:
        print("Structured output FAILED. Raw reply:")
        print(result.raw)
        return None
    print(result.pydantic.model_dump_json(indent=2))
    return result.pydantic

smoke_results = {}
for name in [PRIMARY_MODEL, FALLBACK_MODEL] if RUN_SMOKE_TEST else []:
    try:
        smoke_results[name] = smoke_test(name)
    except Exception as e:   # show the error but keep going, so both models get tested
        print(f"=== {name} === ERROR: {type(e).__name__}: {e}")
        smoke_results[name] = None

In [ ]:
# ✅ Checkpoint 1
for name, report in smoke_results.items():
    print(f"{name}: {'PASS' if isinstance(report, ContentReport) else 'FAIL'}")

if not RUN_SMOKE_TEST:
    print("Smoke test switched off; nothing to check.")
elif isinstance(smoke_results.get(PRIMARY_MODEL), ContentReport):
    print("\nPrimary model works. Keep it for every evaluation run.")
else:
    print("\nPrimary model failed. Stop here and decide on a fallback before building further.")

**✅ Checkpoint 1:** the model names are found, the plain call replies with OK,
and the primary model shows `PASS` with a sensible report (intent `credential_theft`, tactics such as urgency and fear,
quotes copied from the email). Note the token usage. It feeds the budget estimate.

## Stage 2: Data acquisition and licensing

We need public emails **with full headers**, because the Technical Analyst reads headers.
The sources were checked on 2026-10-22:

| Source | Label | Years | Headers | License / terms |
|---|---|---|---|---|
| [Nazario phishing corpus](https://monkey.org/~jose/phishing/) (2023, 2024, 2025 mailboxes) | phishing | 2023–2025 | full | CC-BY-4.0 (attribution: Jose Nazario) |
| [SpamAssassin public corpus](https://spamassassin.apache.org/old/publiccorpus/) (20030228 easy_ham, easy_ham_2, hard_ham) | legitimate | ~2002 | full | No formal license. Offered publicly for testing spam filters. Copyright stays with the original senders. Not redistributed here. |

Sources considered and **not** used:
- **SpamAssassin spam** is spam, not phishing.
- **Enron raw** is 443 MB of 2001 internal mail with no auth headers.
- **The Zenodo curated CSVs** (CC-BY-4.0) have no full headers, label spam rather than phishing, and their Enron copy is lowercased and tokenised.

**Known limitation (era gap).** The phishing is from 2023–2025, while the legitimate mail is from about 2002.
A model can tell them apart by date formats, header styles, or the presence of `Authentication-Results`
headers (modern mail has them, 2002 mail doesn't), rather than by phishing signals. To reduce this:
- the baseline and the LLM-only config get the email with date, routing and auth headers removed;
- the Technical Analyst scores auth headers only when they show an outright `fail`, never just because they are present or missing.

This limitation is stated in the report.

Raw files are downloaded once into Drive. Later runs reuse them. The Nazario server is slow (expect a few minutes).

In [ ]:
import urllib.request, tarfile, mailbox, hashlib
import pandas as pd

RAW_DIR = f"{DRIVE_ROOT}/data/raw"
os.makedirs(RAW_DIR, exist_ok=True)

NAZARIO_FILES = ["phishing-2023", "phishing-2024", "phishing-2025"]
SPAMASSASSIN_FILES = ["20030228_easy_ham.tar.bz2", "20030228_easy_ham_2.tar.bz2", "20030228_hard_ham.tar.bz2"]

def download(url, dest):
    """Download url to dest, unless it is already there."""
    if os.path.exists(dest):
        print("already have", os.path.basename(dest))
        return
    print("downloading", url)
    urllib.request.urlretrieve(url, dest + ".part")
    os.replace(dest + ".part", dest)   # only a complete download gets the real name

for name in NAZARIO_FILES:
    download(f"https://monkey.org/~jose/phishing/{name}", f"{RAW_DIR}/{name}")
for name in SPAMASSASSIN_FILES:
    download(f"https://spamassassin.apache.org/old/publiccorpus/{name}", f"{RAW_DIR}/{name}")

In [ ]:
def make_email_id(raw):
    """Stable ID: the same email text always gets the same ID, across runs and machines."""
    normalised = "\n".join(line.rstrip() for line in raw.replace("\r\n", "\n").split("\n")).strip()
    return hashlib.sha256(normalised.encode("utf-8", errors="replace")).hexdigest()[:16]

def to_text(raw_bytes):
    # Emails are mostly ASCII. "replace" keeps odd bytes from crashing the parse.
    return raw_bytes.decode("utf-8", errors="replace")

rows = []

# Nazario: each file is an mbox (many emails in one file).
for name in NAZARIO_FILES:
    for msg in mailbox.mbox(f"{RAW_DIR}/{name}"):
        if "FOLDER INTERNAL DATA" in str(msg.get("Subject", "")):
            continue   # mail-client bookkeeping record at the top of each file, not an email
        rows.append({"source": f"nazario_{name[-4:]}", "label": 1, "raw": to_text(msg.as_bytes())})

# SpamAssassin: each archive holds one email per file.
for name in SPAMASSASSIN_FILES:
    with tarfile.open(f"{RAW_DIR}/{name}", "r:bz2") as tar:
        for member in tar.getmembers():
            if not member.isfile() or member.name.endswith("cmds"):
                continue   # "cmds" is an index file, not an email
            raw = to_text(tar.extractfile(member).read())
            rows.append({"source": "spamassassin_" + name.split("_", 1)[1].removesuffix(".tar.bz2"),
                         "label": 0, "raw": raw})

emails = pd.DataFrame(rows)
emails["email_id"] = emails["raw"].map(make_email_id)
emails["has_headers"] = emails["raw"].str.contains(r"(?m)^From:", regex=True)   # (?m): ^ matches at every line start
emails["synthetic"] = False
emails = emails[["email_id", "source", "label", "has_headers", "synthetic", "raw"]]

emails.to_parquet(f"{DRIVE_ROOT}/data/emails.parquet", index=False)
print("saved", len(emails), "emails")

In [ ]:
# ✅ Checkpoint 2
print(emails.groupby(["source", "label"]).size().rename("count").to_string(), "\n")
print("phishing:", (emails.label == 1).sum(), "  legitimate:", (emails.label == 0).sum())
print("without headers:", (~emails.has_headers).sum())
print("duplicate IDs (exact copies, removed in Stage 3):", emails.email_id.duplicated().sum(), "\n")

# One parsed sample of each class, to check the parsing looks right
import email
from email import policy
for label in [1, 0]:
    row = emails[emails.label == label].iloc[0]
    msg = email.message_from_string(row.raw, policy=policy.default)
    print(f"--- label {label} ({row.source}, id {row.email_id})")
    print("From:   ", msg["From"])
    print("Subject:", msg["Subject"])
    body = msg.get_body(preferencelist=("plain", "html"))
    print(body.get_content()[:300] if body else "(no text body)", "\n")

**✅ Checkpoint 2:** a count table per source; roughly 1,000+ phishing and about 4,100 legitimate emails;
`without headers` close to 0; and two samples whose From/Subject look like real emails.
The classes are unbalanced, which is fine: Stage 3 samples a balanced 50/50 test subset.

## Stage 3: Dedupe, split, sample

Phishing campaigns send the same email to many people, so a corpus often holds copies that differ only in
spacing, case, or a tracking number. If one copy lands in training and another in testing, the test score is inflated.
So we remove duplicates **before** splitting.

Two emails count as duplicates when their **body text** is the same after normalising: lower-case, digits removed
(which drops tracking numbers and dates), and punctuation and spacing collapsed. This catches copies, not rewrites.

Then:
- **Train / held-out split, 80 / 20.** Stratified by source, so every source appears in both parts in the same proportion. The baseline trains on the train part only.
- **Test subset, 100 emails from the held-out part.** 50 phishing and 50 legitimate, stratified by source. These are the emails all four systems are compared on.
- **The IDs of every split are saved**, so every later run uses exactly the same emails.

In [ ]:
import re, json
import email
from email import policy
from bs4 import BeautifulSoup
from sklearn.model_selection import train_test_split

def get_body_text(raw):
    """The readable text of an email: the plain-text part if there is one, otherwise the HTML part converted to text."""
    try:
        msg = email.message_from_string(raw, policy=policy.default)
        part = msg.get_body(preferencelist=("plain", "html"))
        if part is None:
            return ""
        content = part.get_content()
        if part.get_content_type() == "text/html":
            content = BeautifulSoup(content, "html.parser").get_text(" ")
        return content
    except Exception:   # badly formed emails exist in every corpus; treat them as having no readable body
        return ""

def body_fingerprint(raw):
    text = get_body_text(raw).lower()
    text = re.sub(r"\d+", "", text)          # drop digits: tracking numbers, dates, amounts
    text = re.sub(r"[\W_]+", " ", text)      # punctuation and spacing -> single space
    return hashlib.sha256(text.strip().encode()).hexdigest()

emails = pd.read_parquet(f"{DRIVE_ROOT}/data/emails.parquet")
emails["body_text"] = emails["raw"].map(get_body_text)
emails["fingerprint"] = emails["raw"].map(body_fingerprint)

empty = emails["body_text"].str.strip() == ""
print("emails with no readable body (dropped):", empty.sum())
emails = emails[~empty]

before = len(emails)
emails = emails.drop_duplicates(subset="fingerprint", keep="first")
print("duplicates removed:", before - len(emails))
print("emails left:", len(emails))

In [ ]:
train, heldout = train_test_split(emails, test_size=0.2, stratify=emails["source"], random_state=SEED)

def take_balanced(pool, n):
    """n emails from pool, keeping each source's share the same as in pool."""
    picked, _ = train_test_split(pool, train_size=n, stratify=pool["source"], random_state=SEED)
    return picked

subset = pd.concat([take_balanced(heldout[heldout.label == 1], 50),
                    take_balanced(heldout[heldout.label == 0], 50)])

# Demo emails, picked by reading the subset (choosing examples is not labelling: the labels come from the corpora).
# They come from the subset, so their evaluation results double as the demo's cached backup.
DEMO_IDS = {
    # "eAlert M&T Bank" sent from stoporgo.org, link to angla.cc. SPF and DKIM pass,
    # which shows that passing authentication only proves who sent it, not that it's safe.
    "obvious_phish": "7deba0cab6553440",
    # Sent through the real Google Drive (DKIM, SPF and DMARC all pass, link is drive.google.com),
    # but the display name uses look-alike Cyrillic letters ("Wеllsfаrgo") and Reply-To goes to a .eu domain.
    "subtle_phish": "a62c055b1cf2435e",
    # A genuine 2002 security-advisory mailing list: security wording and links, but consistent domains.
    "legitimate": "12865f8f440c21cc",
}

splits = {"train": train.email_id.tolist(),
          "heldout": heldout.email_id.tolist(),
          "subset": subset.email_id.tolist(),
          "demo": DEMO_IDS}
with open(f"{DRIVE_ROOT}/data/splits.json", "w") as f:
    json.dump(splits, f)

# Keep the deduplicated table too; later stages load it with the splits.
emails.drop(columns=["fingerprint"]).to_parquet(f"{DRIVE_ROOT}/data/emails_dedup.parquet", index=False)

In [ ]:
# ✅ Checkpoint 3
balance = pd.DataFrame({
    "train":   train.groupby("source").size(),
    "heldout": heldout.groupby("source").size(),
    "subset":  subset.groupby("source").size(),
}).fillna(0).astype(int)
print(balance, "\n")
print("subset labels:", subset.label.value_counts().to_dict())

overlap = set(splits["train"]) & set(splits["heldout"])
print("train/held-out overlap:", len(overlap))
print("subset inside held-out:", set(splits["subset"]) <= set(splits["heldout"]))
assert not overlap and set(splits["subset"]) <= set(splits["heldout"])

demo_ok = set(DEMO_IDS.values()) <= set(splits["subset"])
print("demo emails inside subset:", demo_ok)
assert demo_ok, "a demo ID is no longer in the subset (did the data or SEED change?)"
print("saved:", f"{DRIVE_ROOT}/data/splits.json")

**✅ Checkpoint 3:** a balance table where every source appears in train, held-out and subset;
the subset is exactly 50 phishing and 50 legitimate; the overlap is 0; the 3 demo emails are inside the subset.

Things noticed while reading the subset, for the report:
- A few phishing emails are in Spanish or German. Multilingual phishing is out of scope, but these are kept because they are part of the labelled corpus.
- Many phishing emails target the corpus owner's own domain (monkey.org), often with a forged `From:` on that domain.
- Most legitimate emails are 2002 technical mailing-list posts, a narrower kind of mail than a normal inbox.

## Stage 4: The Technical Analyst's tools

These are plain Python functions with no LLM, so they run offline and give the same answer every time.
Each check returns **findings**: a check name, a severity (low / medium / high), a plain-English detail,
and the exact evidence (a header value or URL).

| Check | What it looks for |
|---|---|
| `reply_to_mismatch` | Replies go to a different domain than the sender's |
| `return_path_mismatch` | Bounces go to a different domain than the sender's (weak signal: bulk mailers do this legitimately) |
| `display_name_domain` | The display name claims a domain, e.g. `"monkey.org" <admin@other.com>` |
| `display_name_brand` | The display name names a known brand, but the sender isn't that brand's domain |
| `mixed_script_name` | A word in the display name mixes Latin letters with look-alike Cyrillic or Greek ones |
| `auth_fail` | SPF, DKIM or DMARC explicitly **failed**. Passing or missing results are not counted (see the era gap in Stage 2) |
| `link_text_mismatch` | A link's visible text shows one domain but it actually goes to another |
| `brand_in_url` | A brand name sits in a link's host but the link isn't on that brand's domain, e.g. `paypal.com.secure-login.xyz` |
| `lookalike_domain` | A domain is one or two letters away from a brand, e.g. `paypall.com` |
| `punycode_domain` | A domain written in punycode (`xn--`), which can hide look-alike characters |
| `url_shortener` | A link goes through a shortener, hiding the real destination |
| `raw_ip_url` | A link goes to a bare IP address instead of a domain |
| `unusual_tld` | A link ends in a top-level domain that is heavily abused, e.g. `.xyz`, `.top` |

Mailing lists legitimately rewrite Reply-To and Return-Path, so those two checks are skipped when the email has a `List-Id` header.

**Score:** for each check, take its most severe finding (high = 35, medium = 20, low = 10) and add them up,
capped at 100. Counting each check once stops one email with twenty bad links from scoring 700.
The weights were chosen by hand and are **not** tuned on the test subset.

In [ ]:
import ipaddress, unicodedata
from dataclasses import dataclass, field
from email.utils import parseaddr
from urllib.parse import urlparse
import tldextract

_extract = tldextract.TLDExtract(suffix_list_urls=())   # use the bundled public-suffix list; no network

def org_domain(host):
    """The registrable domain: 'mail.secure.paypal.co.uk' -> 'paypal.co.uk'. IP addresses come back unchanged."""
    host = (host or "").lower().strip(".")
    ext = _extract(host)
    return f"{ext.domain}.{ext.suffix}" if ext.suffix else host

def url_host(url):
    try:
        return (urlparse(url).hostname or "").lower()
    except ValueError:   # malformed URLs exist in phishing mail
        return ""

URL_RE = re.compile(r"""https?://[^\s"'<>()\[\]]+""", re.I)

@dataclass
class ParsedEmail:
    from_name: str
    from_domain: str
    reply_to_domain: str
    return_path_domain: str
    list_id: str
    auth: dict                 # {"spf": "pass", "dkim": "none", "dmarc": "fail"}
    auth_available: bool
    subject: str
    body_text: str
    anchors: list = field(default_factory=list)   # (visible text, href) pairs from the HTML part
    urls: list = field(default_factory=list)

def _addr_domain(value):
    return org_domain(parseaddr(value)[1].rpartition("@")[2])

def parse_auth(auth_header, received_spf):
    """Read spf / dkim / dmarc results. 'none' means the header doesn't mention it."""
    results = {}
    for mech in ("spf", "dkim", "dmarc"):
        m = re.search(rf"\b{mech}=(\w+)", auth_header, re.I)
        results[mech] = m.group(1).lower() if m else "none"
    if results["spf"] == "none" and received_spf:
        results["spf"] = received_spf.split()[0].lower()   # "Received-SPF: fail (...)" -> "fail"
    return results

def parse_email(raw):
    msg = email.message_from_string(raw, policy=policy.default)

    def header(name):
        try:
            return str(msg.get(name, "") or "")
        except Exception:   # a malformed header shouldn't stop the analysis
            return ""

    plain, html = [], []
    for part in msg.walk():
        if part.is_multipart() or part.get_content_disposition() == "attachment":
            continue
        if part.get_content_type() in ("text/plain", "text/html"):
            try:
                content = part.get_content()
            except Exception:
                continue
            (plain if part.get_content_type() == "text/plain" else html).append(content)

    soup = BeautifulSoup("\n".join(html), "html.parser") if html else None
    if soup:
        for tag in soup(["style", "script"]):   # CSS and scripts aren't readable text
            tag.decompose()
    anchors = [(a.get_text(" ", strip=True), a["href"].strip()) for a in soup.find_all("a", href=True)] if soup else []
    body_text = "\n".join(plain) if plain else (soup.get_text(" ") if soup else "")
    urls = URL_RE.findall(body_text) + [href for _, href in anchors if href.lower().startswith("http")]

    # The first Authentication-Results header is the one added by the receiving server.
    auth_header, received_spf = header("Authentication-Results"), header("Received-SPF")
    return ParsedEmail(
        from_name=parseaddr(header("From"))[0],
        from_domain=_addr_domain(header("From")),
        reply_to_domain=_addr_domain(header("Reply-To")),
        return_path_domain=_addr_domain(header("Return-Path")),
        list_id=header("List-Id"),
        auth=parse_auth(auth_header, received_spf),
        auth_available=bool(auth_header or received_spf),
        subject=header("Subject"),
        body_text=body_text,
        anchors=anchors,
        urls=list(dict.fromkeys(urls)),   # remove repeats, keep order
    )

In [ ]:
from typing import Literal
from pydantic import BaseModel

class TechFinding(BaseModel):
    check: str
    severity: Literal["low", "medium", "high"]
    detail: str
    evidence: str

class TechReport(BaseModel):
    findings: list[TechFinding]
    tech_score: int            # 0-100
    auth_available: bool
    auth: dict[str, str]
    from_domain: str
    url_domains: list[str]

### Reference lists

These are small hand-made lists, not complete ones. Each brand maps to the domains it really sends from.
The abused-TLD list follows the top-level domains that repeatedly lead abuse rankings in public reports
(for example Spamhaus and Interisle), and is meant as an illustration rather than a complete list.

In [ ]:
BRANDS = {
    "paypal": {"paypal.com"},
    "apple": {"apple.com", "icloud.com"},
    "icloud": {"icloud.com", "apple.com"},
    "microsoft": {"microsoft.com", "live.com", "outlook.com", "office.com", "microsoftonline.com", "sharepoint.com"},
    "outlook": {"microsoft.com", "live.com", "outlook.com", "office.com", "microsoftonline.com"},
    "sharepoint": {"sharepoint.com", "microsoft.com"},
    "onedrive": {"live.com", "microsoft.com", "sharepoint.com"},
    "google": {"google.com", "googlemail.com", "gmail.com"},
    "gmail": {"google.com", "googlemail.com", "gmail.com"},
    "amazon": {"amazon.com", "amazon.co.uk", "amazon.de", "amazon.in", "amazon.ca", "amazonses.com"},
    "netflix": {"netflix.com"},
    "facebook": {"facebook.com", "facebookmail.com", "meta.com"},
    "instagram": {"instagram.com"},
    "whatsapp": {"whatsapp.com"},
    "linkedin": {"linkedin.com"},
    "dropbox": {"dropbox.com", "dropboxmail.com"},
    "docusign": {"docusign.com", "docusign.net"},
    "adobe": {"adobe.com"},
    "wellsfargo": {"wellsfargo.com"},
    "chase": {"chase.com", "jpmorgan.com"},
    "bankofamerica": {"bankofamerica.com", "bofa.com"},
    "citibank": {"citibank.com", "citi.com"},
    "hsbc": {"hsbc.com", "hsbc.co.uk"},
    "barclays": {"barclays.com", "barclays.co.uk"},
    "americanexpress": {"americanexpress.com", "aexp.com"},
    "fedex": {"fedex.com"},
    "dhl": {"dhl.com", "dhl.de"},
    "ups": {"ups.com"},
    "usps": {"usps.com"},
    "irs": {"irs.gov"},
    "coinbase": {"coinbase.com"},
    "metamask": {"metamask.io"},
    "binance": {"binance.com"},
    "spotify": {"spotify.com"},
    "ebay": {"ebay.com"},
    "walmart": {"walmart.com"},
    "ringcentral": {"ringcentral.com"},
    "cpanel": {"cpanel.net", "cpanel.com"},
    "icann": {"icann.org"},
}

SHORTENERS = {"bit.ly", "tinyurl.com", "t.co", "goo.gl", "ow.ly", "is.gd", "buff.ly", "rebrand.ly",
              "cutt.ly", "shorturl.at", "rb.gy", "tiny.cc", "s.id", "t.ly", "bit.do"}

UNUSUAL_TLDS = {"xyz", "top", "club", "online", "site", "shop", "icu", "buzz", "tk", "ml", "ga", "cf", "gq",
                "zip", "mov", "click", "link", "live", "rest", "cyou", "sbs", "cfd", "bond", "pw", "su"}

# Characters attackers use to imitate Latin letters, mapped to the letter they imitate.
LOOKALIKE_CHARS = str.maketrans({
    "а": "a", "е": "e", "о": "o", "р": "p", "с": "c", "у": "y", "х": "x", "і": "i", "ѕ": "s", "ј": "j",
    "ԁ": "d", "ӏ": "l", "ο": "o", "α": "a", "ν": "v", "ρ": "p", "ε": "e",
    "0": "o", "1": "l", "3": "e", "5": "s",
})

def normalise(text):
    """Lower-case, map look-alike characters to Latin letters, strip accents, undo 'rn'->'m' and 'vv'->'w' tricks."""
    text = text.lower().translate(LOOKALIKE_CHARS)
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return text.replace("rn", "m").replace("vv", "w")

def brand_hits(text):
    """Brands named in text: as a separate word, or (for names of 6+ letters) anywhere once spaces are removed,
    so 'Wells Fargo' still matches 'wellsfargo'."""
    norm = normalise(text)
    words = set(re.split(r"[^a-z0-9]+", norm))
    joined = re.sub(r"[^a-z0-9]", "", norm)
    return {b for b in BRANDS if b in words or (len(b) >= 6 and b in joined)}

def edit_distance(a, b):
    """Levenshtein distance: the fewest single-letter edits that turn a into b."""
    prev = list(range(len(b) + 1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (ca != cb)))
        prev = cur
    return prev[-1]

In [ ]:
def check_headers(p):
    findings = []
    on_mailing_list = bool(p.list_id)   # lists rewrite Reply-To and Return-Path legitimately
    if p.reply_to_domain and p.from_domain and p.reply_to_domain != p.from_domain and not on_mailing_list:
        findings.append(TechFinding(check="reply_to_mismatch", severity="medium",
            detail="Replies would go to a different domain than the one the email claims to come from.",
            evidence=f"From: {p.from_domain} / Reply-To: {p.reply_to_domain}"))
    if p.return_path_domain and p.from_domain and p.return_path_domain != p.from_domain and not on_mailing_list:
        findings.append(TechFinding(check="return_path_mismatch", severity="low",
            detail="Bounces go to a different domain than the sender's. Bulk mailers also do this, so it is a weak signal.",
            evidence=f"From: {p.from_domain} / Return-Path: {p.return_path_domain}"))

    for claimed in re.findall(r"[a-z0-9-]+(?:\.[a-z0-9-]+)*\.[a-z]{2,}", p.from_name.lower()):
        ext = _extract(claimed)
        if not ext.suffix or len(ext.domain) < 2:
            continue   # not a real domain, e.g. the initial in "Craig R.Hughes" (".hughes" is a real TLD!)
        if org_domain(claimed) != p.from_domain:
            findings.append(TechFinding(check="display_name_domain", severity="medium",
                detail="The sender's display name names one domain, but the email comes from another.",
                evidence=f'"{p.from_name}" <...@{p.from_domain}>'))
            break

    for brand in sorted(brand_hits(p.from_name)):
        if p.from_domain not in BRANDS[brand]:
            findings.append(TechFinding(check="display_name_brand", severity="high",
                detail=f"The display name suggests {brand}, but the email doesn't come from {brand}'s domain.",
                evidence=f'"{p.from_name}" <...@{p.from_domain}>'))

    for word in p.from_name.split():
        scripts = {unicodedata.name(ch, "").split()[0] for ch in word if ch.isalpha()}
        if "LATIN" in scripts and scripts & {"CYRILLIC", "GREEK"}:
            findings.append(TechFinding(check="mixed_script_name", severity="high",
                detail="The display name mixes normal letters with look-alike Cyrillic or Greek ones, a disguise trick.",
                evidence=word))
            break
    return findings

def check_auth(p):
    a, findings = p.auth, []
    if a["spf"] in ("fail", "softfail"):
        findings.append(TechFinding(check="auth_fail", severity="high" if a["spf"] == "fail" else "medium",
            detail="SPF failed: the sending server isn't authorised to send for this domain.", evidence=f"spf={a['spf']}"))
    if a["dkim"] == "fail":
        findings.append(TechFinding(check="auth_fail", severity="medium",
            detail="DKIM failed: the email's signature doesn't match, so it may have been altered or forged.",
            evidence="dkim=fail"))
    if a["dmarc"] == "fail":
        findings.append(TechFinding(check="auth_fail", severity="high",
            detail="DMARC failed: the From domain's own policy says this email is not genuine.", evidence="dmarc=fail"))
    return findings

DOMAIN_IN_TEXT = re.compile(r"(?:https?://)?((?:[a-z0-9-]+\.)+[a-z]{2,})", re.I)

def check_links(p):
    findings = []
    for text, href in p.anchors:
        m = DOMAIN_IN_TEXT.fullmatch(text.strip().rstrip("/"))   # the visible text is only an address
        href_dom = org_domain(url_host(href))
        if m and href_dom and org_domain(m.group(1)) != href_dom:
            findings.append(TechFinding(check="link_text_mismatch", severity="high",
                detail="A link shows one address but actually goes somewhere else.",
                evidence=f"shows '{text[:60]}' -> goes to {href_dom}"))
    return findings

def check_urls(p):
    findings = []
    hosts = list(dict.fromkeys(h for h in (url_host(u) for u in p.urls) if h))
    for host in hosts:
        org = org_domain(host)
        try:
            ipaddress.ip_address(host)
            findings.append(TechFinding(check="raw_ip_url", severity="high",
                detail="A link goes to a bare IP address instead of a named website.", evidence=host))
            continue
        except ValueError:
            pass
        if org in SHORTENERS:
            findings.append(TechFinding(check="url_shortener", severity="medium",
                detail="A link uses a URL shortener, which hides where it really goes.", evidence=host))
        if org.rsplit(".", 1)[-1] in UNUSUAL_TLDS:
            findings.append(TechFinding(check="unusual_tld", severity="low",
                detail="A link uses a top-level domain that is heavily abused by phishing sites.", evidence=host))
        if "xn--" in host:
            findings.append(TechFinding(check="punycode_domain", severity="medium",
                detail="A link's domain is written in punycode, which can hide look-alike characters.", evidence=host))
        for brand in sorted(brand_hits(host.removesuffix(org) + _extract(org).domain)):
            if org not in BRANDS[brand]:
                findings.append(TechFinding(check="brand_in_url", severity="high",
                    detail=f"A link mentions {brand} but doesn't go to {brand}'s real domain.", evidence=host))
    return findings

def check_lookalikes(p):
    """Domains (sender's and links') that are one or two edits away from a brand name."""
    findings = []
    domains = [p.from_domain] + [org_domain(url_host(u)) for u in p.urls]
    for org in dict.fromkeys(d for d in domains if d):
        label = normalise(_extract(org).domain)
        for brand, official in BRANDS.items():
            if len(brand) < 5 or org in official:
                continue   # short names like "ups" give too many accidental near-matches
            if 0 < edit_distance(label, brand) <= (1 if len(brand) <= 7 else 2):
                findings.append(TechFinding(check="lookalike_domain", severity="high",
                    detail=f"The domain looks like {brand}'s but isn't.", evidence=org))
    return findings

WEIGHTS = {"high": 35, "medium": 20, "low": 10}

def tech_score(findings):
    worst = {}
    for f in findings:   # count each check once, at its most severe finding
        worst[f.check] = max(worst.get(f.check, 0), WEIGHTS[f.severity])
    return min(100, sum(worst.values()))

def analyze_technical(raw):
    p = parse_email(raw)
    findings = check_headers(p) + check_auth(p) + check_links(p) + check_urls(p) + check_lookalikes(p)
    return TechReport(findings=findings, tech_score=tech_score(findings), auth_available=p.auth_available,
                      auth=p.auth, from_domain=p.from_domain,
                      url_domains=list(dict.fromkeys(org_domain(url_host(u)) for u in p.urls if url_host(u))))

def score_to_label(score):
    if score >= PHISHING_THRESHOLD:
        return "phishing"
    return "suspicious" if score >= SUSPICIOUS_THRESHOLD else "legitimate"

### Tests on tiny made-up emails

Each check gets a case where it should fire and, where useful, one where it shouldn't.

In [ ]:
def make_test_email(sender, body, html=False, reply_to=None, auth=None, list_id=None):
    lines = [f"From: {sender}", "To: reader@example.com", "Subject: test"]
    if reply_to: lines.append(f"Reply-To: {reply_to}")
    if auth:     lines.append(f"Authentication-Results: mx.example.com; {auth}")
    if list_id:  lines.append(f"List-Id: {list_id}")
    lines += ["MIME-Version: 1.0", f"Content-Type: text/{'html' if html else 'plain'}; charset=utf-8", "", body]
    return "\n".join(lines)

def checks_fired(raw):
    return {f.check for f in analyze_technical(raw).findings}

# Helpers
assert org_domain("mail.secure.paypal.co.uk") == "paypal.co.uk"
assert edit_distance("paypall", "paypal") == 1

# A clean email has no findings and a score of 0
clean = make_test_email("Alice <alice@example.com>", "Notes: https://example.com/page", auth="spf=pass dkim=pass")
assert checks_fired(clean) == set() and analyze_technical(clean).tech_score == 0

# Headers
assert "reply_to_mismatch" in checks_fired(make_test_email("a@example.com", "hi", reply_to="x@other.net"))
assert "reply_to_mismatch" not in checks_fired(make_test_email("a@example.com", "hi", reply_to="x@other.net", list_id="<l.example.org>"))
assert "display_name_domain" in checks_fired(make_test_email('"monkey.org" <admin@angsaraya.com>', "hi"))
assert "display_name_domain" not in checks_fired(make_test_email('"Craig R.Hughes" <craig@example.com>', "hi"))
assert "display_name_brand" in checks_fired(make_test_email("PayPal Service <help@secure-mail.net>", "hi"))
assert "display_name_brand" not in checks_fired(make_test_email("PayPal <service@paypal.com>", "hi"))
assert "mixed_script_name" in checks_fired(make_test_email("Wеllsfаrgo Bаnk <x@example.com>", "hi"))  # Cyrillic е, а

# Authentication: only explicit failures count
assert "auth_fail" in checks_fired(make_test_email("a@example.com", "hi", auth="spf=fail dkim=none"))
assert "auth_fail" not in checks_fired(make_test_email("a@example.com", "hi", auth="spf=pass dkim=pass dmarc=pass"))

# Links and URLs
assert "link_text_mismatch" in checks_fired(make_test_email("a@example.com",
    '<a href="http://evil-site.top/x">www.paypal.com</a>', html=True))
assert "link_text_mismatch" not in checks_fired(make_test_email("a@example.com",
    '<a href="https://www.paypal.com/signin">www.paypal.com</a>', html=True))
assert "brand_in_url" in checks_fired(make_test_email("a@example.com", "http://paypal.com.secure-login.xyz/"))
assert "brand_in_url" in checks_fired(make_test_email("a@example.com", "http://paypa1.com/login"))
assert "lookalike_domain" in checks_fired(make_test_email("a@example.com", "http://paypall.com/login"))
assert not checks_fired(make_test_email("a@example.com", "https://www.paypal.com/"))
assert "url_shortener" in checks_fired(make_test_email("a@example.com", "https://bit.ly/abc"))
assert "raw_ip_url" in checks_fired(make_test_email("a@example.com", "http://192.168.10.5/login"))
assert "punycode_domain" in checks_fired(make_test_email("a@example.com", "http://xn--pypal-4ve.com/"))
assert "unusual_tld" in checks_fired(make_test_email("a@example.com", "http://example.xyz/"))

print("all tool tests passed")

### Run the tools on the data

On the **train** split we only look at how often each check fires for each class. That tells us which checks are
useful, and shows signs of the era gap: a check that never fires on 2002 mail may be detecting the era,
not phishing. The **subset** gets the tools-only metrics. Nothing is tuned on it.

In [ ]:
from sklearn.metrics import precision_recall_fscore_support

def evaluate(y_true, verdicts):
    """Binary metrics with phishing as the positive class. 'suspicious' counts as a phishing prediction,
    and 'error' (the analysis failed) counts as a wrong prediction."""
    y_pred = [1 - y if v == "error" else int(v != "legitimate") for y, v in zip(y_true, verdicts)]
    p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    return {"precision": round(p, 3), "recall": round(r, 3), "f1": round(f, 3)}

def confusion_table(y_true, verdicts):
    truth = pd.Series(["phishing" if y else "legitimate" for y in y_true], name="true label")
    pred = pd.Series(list(verdicts), name="verdict")
    return pd.crosstab(truth, pred).reindex(columns=["phishing", "suspicious", "legitimate", "error"], fill_value=0)

emails = pd.read_parquet(f"{DRIVE_ROOT}/data/emails_dedup.parquet").set_index("email_id")
with open(f"{DRIVE_ROOT}/data/splits.json") as f:
    splits = json.load(f)

tech_reports = {eid: analyze_technical(emails.at[eid, "raw"]) for eid in splits["train"] + splits["subset"]}
print("analysed", len(tech_reports), "emails")

# How often each check fires, per class, on the train split
fired = pd.DataFrame([{"label": emails.at[eid, "label"], **{f.check: 1 for f in tech_reports[eid].findings}}
                      for eid in splits["train"]]).fillna(0)
rates = (fired.groupby("label").mean() * 100).T.round(1)
rates.columns = ["% of legitimate", "% of phishing"]
print(rates.sort_values("% of phishing", ascending=False))

In [ ]:
# ✅ Checkpoint 4: tools-only results on the 100-email subset
subset_ids = splits["subset"]
y_true = [emails.at[eid, "label"] for eid in subset_ids]
tools_verdicts = [score_to_label(tech_reports[eid].tech_score) for eid in subset_ids]

print("Tools only:", evaluate(y_true, tools_verdicts))
print(confusion_table(y_true, tools_verdicts), "\n")

for role, eid in splits["demo"].items():
    report = tech_reports[eid]
    print(f"--- {role}: score {report.tech_score} -> {score_to_label(report.tech_score)}")
    for f in report.findings:
        print(f"   [{f.severity}] {f.check}: {f.evidence}")

**✅ Checkpoint 4:** `all tool tests passed`, a table of how often each check fires per class,
and tools-only precision / recall / F1 with a confusion matrix. This stage makes no API calls.

The tools-only system is expected to have **high precision but low recall**: it only sees technical tricks,
and much phishing (for example a plain request sent from a free webmail account) has none.
That gap is what the Content Analyst is for.

Notes for the report:
- **`auth_fail` fires on 0% of legitimate mail, but partly for the wrong reason.** 2002 mail has no authentication
  headers at all, so it *can't* fail. Its high precision here is inflated by the era gap.
  The same applies, more weakly, to `unusual_tld`: most of those TLDs didn't exist in 2002.
- **The "obvious" demo phish scores 0 with the tools.** A human spots it at once from its wording, but technically
  it is clean: it passes SPF and DKIM, has no Reply-To, and "M&T Bank" isn't in the brand list. The "subtle" one,
  which looks fine to a human, scores 90. The two analysts see different things, which is exactly when the
  disagreement round should fire.
- A few checks were corrected after looking at false alarms on **train** legitimate mail (never the test subset):
  a display name like "Craig R.Hughes" read as the domain `r.hughes`, a link heading like "News.com: Top CIOs"
  read as an address, and `netli.com` matching netflix.

## Stage 5: Baseline (TF-IDF + logistic regression)

A classic text classifier, used as the yardstick for the agent system.
- **TF-IDF** turns each email into a vector of word weights. Words that are frequent in one email but rare
  across the corpus get high weight.
- **Logistic regression** learns one weight per word (and per word pair) and adds them up into a phishing probability.

It trains on the **train** split and is tested on the full held-out split and on the 100-email subset.

**Keeping it honest.** A classifier happily learns any shortcut that separates the two corpora, so the input is cleaned first:
- **Subject and body text only, no headers.** This removes dates, mail servers and authentication headers (the era gap).
- **No digits.** Years like "2002" or "© 2024" would give the era away.
- **The phishing corpus owner's name and address are removed.** Nearly every Nazario email is addressed to
  `jose@monkey.org`, so "monkey" alone would be a near-perfect but meaningless phishing word. (A first version
  replaced it with a placeholder word, and the placeholder became the top phishing feature, so it is now deleted.)
- **Links become the word `URL`.** Otherwise `https` (modern) against `http` (2002) gives the era away.
  Judging the links themselves is the Technical Analyst's job.

The probability is turned into a 0–100 score and mapped to a verdict with the same thresholds as everything else.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

def baseline_text(raw):
    p = parse_email(raw)
    text = f"{p.subject}\n{p.body_text}"
    text = URL_RE.sub(" URL ", text)                              # http vs https would give the era away
    text = re.sub(r"\S*www\.\S+", " URL ", text, flags=re.I)        # bare www.example.com addresses too
    return re.sub(r"jose@monkey\.org|monkey\.org|\bmonkey\b|\bjose\b", " ", text, flags=re.I)   # corpus owner

train_ids, heldout_ids = splits["train"], splits["heldout"]
X_train = [baseline_text(emails.at[eid, "raw"]) for eid in train_ids]
y_train = [emails.at[eid, "label"] for eid in train_ids]

vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b[^\W\d_]{2,}\b",   # words of 2+ letters; digits never become features
    ngram_range=(1, 2),                       # single words and word pairs
    min_df=2,                                 # ignore words seen in only one email
    sublinear_tf=True,                        # a word repeated 50 times isn't 50 times more important
)
classifier = LogisticRegression(max_iter=2000, class_weight="balanced")   # "balanced": there's ~3x more legitimate mail
classifier.fit(vectorizer.fit_transform(X_train), y_train)
print("vocabulary size:", len(vectorizer.vocabulary_))

def baseline_scores(ids):
    X = vectorizer.transform([baseline_text(emails.at[eid, "raw"]) for eid in ids])
    return [round(p * 100) for p in classifier.predict_proba(X)[:, 1]]

In [ ]:
# ✅ Checkpoint 5
heldout_verdicts = [score_to_label(s) for s in baseline_scores(heldout_ids)]
print("Baseline, full held-out split:", evaluate([emails.at[eid, "label"] for eid in heldout_ids], heldout_verdicts))

baseline_verdicts = [score_to_label(s) for s in baseline_scores(subset_ids)]
print("Baseline, 100-email subset:  ", evaluate(y_true, baseline_verdicts))
print(confusion_table(y_true, baseline_verdicts), "\n")

# The words the classifier relies on most. Era or corpus artefacts here would be a warning sign.
words = vectorizer.get_feature_names_out()
order = classifier.coef_[0].argsort()
print("Most 'phishing' features:   ", ", ".join(words[order[-25:][::-1]]))
print("Most 'legitimate' features: ", ", ".join(words[order[:25]]))

**✅ Checkpoint 5:** baseline metrics on the full held-out split and on the subset, a confusion matrix,
and the classifier's top features. This stage makes no API calls.

**Reading the result.** The baseline is near-perfect (F1 about 0.97 on held-out, 0.99 on the subset), but look at *why*:
- Its phishing words are genuine phishing language: *your account, password, payment, invoice, mailbox, action required*.
- Its legitimate words mostly describe **2002 technical mailing lists**: *wrote, linux, mailing list, rpm*,
  and *url date* (from RSS-feed emails). It has largely learned "mailing-list post vs. not", which works here
  only because almost every legitimate email in this corpus is one.

So this dataset is easy for a word-counting model, and that tells us more about the dataset than about phishing.
A modern legitimate email (a real password-reset notice or an invoice) would likely fool it.
The agent system's case therefore rests less on beating this F1 and more on what the baseline can't do:
explain its decision with evidence, and work without training data.

## Stage 6: LLM plumbing (throttle, retry, cache, daily counter)

Every LLM call in the rest of the notebook goes through one function, `run_step`. It does five things, in order:

1. **Cache.** If this exact step (model, prompt version, configuration, email, step name) has run before,
   it loads the saved answer from disk and makes no API call. An interrupted run therefore resumes where it stopped.
   Files are written atomically (to a `.tmp` file, then renamed), so a disconnect can't leave half a file.
2. **Daily budget.** It refuses to call once today's count reaches `RPD_SAFETY_STOP`. "Today" is the Pacific-time date,
   because that's when Google resets the quota.
3. **Throttle.** It waits until fewer than `THROTTLE_RPM` calls were made in the last 60 seconds.
4. **Retry.** On a rate-limit (429) or server error (500/503) it waits 5, 10, 20, 40, 60, 60... s (plus a little randomness)
   and tries again, up to `MAX_RETRIES` times. "Server busy" (503) is common on the free tier at peak times.
   If Google says the **daily** quota is used up, it stops at once, because retrying can't help until midnight Pacific.
5. **Parse.** It takes CrewAI's parsed object or, failing that, digs the JSON out of the raw reply. If that fails too,
   it asks the model **once** to repair its answer. If the repair also fails, the step is saved as failed (`None`)
   instead of crashing the run. To retry a failed step, delete its cache file.

CrewAI agents normally retry failed tasks by themselves. That is switched off (`max_retry_limit=0`),
so every request is counted here and nowhere else.

The daily counter lives next to the cache, so a local run and a Colab run keep **separate** counts,
while Google's real limit covers both. If they ever disagree, Google's own "daily quota exhausted" error still stops the run.

In [ ]:
import time, random, datetime
from zoneinfo import ZoneInfo

CACHE_DIR = f"{DRIVE_ROOT}/cache"

class DailyBudgetReached(Exception):
    """Today's request budget is used up. Resume after midnight Pacific time."""

class CacheMiss(Exception):
    """USE_CACHE_ONLY is on and this step has no cached result."""

def write_json(path, data):
    """Write JSON atomically: a crash mid-write leaves the old file (or none), never half a file."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path + ".tmp", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=1)
    os.replace(path + ".tmp", path)

# ---- Daily request counter, per model (Google's quotas are per model), keyed on the Pacific-time date ----
def usage_path():
    today = datetime.datetime.now(ZoneInfo("America/Los_Angeles")).date().isoformat()
    return f"{CACHE_DIR}/usage_{PRIMARY_MODEL}_{today}.json"

def requests_today():
    try:
        with open(usage_path()) as f:
            return json.load(f)["requests"]
    except FileNotFoundError:
        return 0

def add_requests(n):
    write_json(usage_path(), {"requests": requests_today() + n})

# ---- Throttle: at most THROTTLE_RPM calls in any 60-second window ----
_recent_calls = []

def wait_for_slot():
    while True:
        now = time.monotonic()
        _recent_calls[:] = [t for t in _recent_calls if now - t < 60]
        if len(_recent_calls) < THROTTLE_RPM:
            _recent_calls.append(now)
            return
        time.sleep(60 - (now - _recent_calls[0]) + 0.1)

# ---- Which errors are worth retrying ----
def error_kind(e):
    """'daily' if the daily quota is exhausted, 'retry' for rate-limit or server errors, None otherwise."""
    chain = []
    while e is not None and len(chain) < 10:   # CrewAI may wrap the original Gemini error; look at the whole chain
        chain.append(e)
        e = e.__cause__ or e.__context__
    text = " ".join(f"{getattr(x, 'code', '')} {x}" for x in chain)
    if "PerDay" in text or "per day" in text.lower():
        return "daily"
    codes = {getattr(x, "code", None) for x in chain}
    if codes & {429, 500, 503} or any(s in text for s in ("429", "RESOURCE_EXHAUSTED", "UNAVAILABLE")):
        return "retry"
    return None

def server_busy(e):
    return "503" in str(e) or "UNAVAILABLE" in str(e) or getattr(e, "code", None) == 503

def call_with_retries(build, inputs):
    """Kick off a freshly built crew, retrying rate-limit and server errors with growing waits."""
    for attempt in range(MAX_RETRIES + 1):
        if requests_today() >= RPD_SAFETY_STOP:
            raise DailyBudgetReached(f"{requests_today()} requests used today (limit {RPD_SAFETY_STOP}). "
                                     "Resume after midnight Pacific time.")
        wait_for_slot()
        try:
            result = run_crew(build(), inputs)
        except Exception as e:
            kind = error_kind(e)
            if not server_busy(e):
                add_requests(1)   # count failed attempts too, to stay on the safe side ...
            # ... except "server busy" (503) rejections, which Google most likely doesn't charge for.
            # If it does, Google's own daily-quota error still stops the run.
            if kind == "daily":
                raise DailyBudgetReached("Google reports the daily quota is used up. "
                                         "Resume after midnight Pacific time.") from e
            if kind != "retry" or attempt == MAX_RETRIES:
                raise
            delay = min(60, 5 * 2 ** attempt) + random.uniform(0, 2)
            print(f"   {type(e).__name__}: retrying in {delay:.0f}s ({attempt + 1}/{MAX_RETRIES})")
            time.sleep(delay)
            continue
        add_requests(max(1, result.token_usage.successful_requests))
        return result

In [ ]:
# ---- Parsing the model's answer ----
def parse_json_text(text, model):
    """Find the JSON object in a reply (even inside ``` fences or surrounded by prose) and validate it."""
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start, end = text.find("{"), text.rfind("}")
    if start == -1 or end <= start:
        raise ValueError("no JSON object in the reply")
    return model.model_validate_json(text[start:end + 1])   # pydantic's ValidationError is a ValueError

def parse_output(result, model):
    if isinstance(result.pydantic, model):
        return result.pydantic
    return parse_json_text(result.raw or "", model)

# ---- Building agents and single-task crews ----
def make_agent(role, goal, backstory):
    return Agent(role=role, goal=goal, backstory=backstory, llm=make_llm(PRIMARY_MODEL),
                 allow_delegation=False,   # agents never hand work to each other on their own
                 max_iter=1,               # one attempt per task
                 max_retry_limit=0,        # no hidden retries; call_with_retries does that, and counts them
                 verbose=False)

def single_task_crew(agent, description, model):
    task = Task(description=description, expected_output=f"A {model.__name__}, as JSON.",
                agent=agent, output_pydantic=model)
    return Crew(agents=[agent], tasks=[task], verbose=False)

# ---- The one function every LLM step goes through ----
def run_step(config, email_id, step, build, inputs, model):
    """Run one cached LLM step. build(note) must return a fresh Crew; note is extra text for the task
    (empty normally, a repair instruction on the second try). Returns (output or None if it failed, new requests)."""
    path = f"{CACHE_DIR}/{PRIMARY_MODEL}/{PROMPT_VERSION}/{config}/{email_id}/{step}.json"
    if os.path.exists(path):
        with open(path, encoding="utf-8") as f:
            cached = json.load(f)
        return (model.model_validate(cached["output"]) if cached["output"] is not None else None), 0
    if USE_CACHE_ONLY:
        raise CacheMiss(path)

    before = requests_today()
    result = call_with_retries(lambda: build(""), inputs)
    raw, error, output = result.raw, None, None
    try:
        output = parse_output(result, model)
    except ValueError as e:
        # One repair attempt. Braces are swapped out because CrewAI treats {name} in a task as a placeholder.
        note = (f"\n\nYour previous reply could not be used ({str(e)[:300]}). Reply again with ONLY a JSON object "
                "that matches the required schema.").replace("{", "(").replace("}", ")")
        result = call_with_retries(lambda: build(note), inputs)
        raw = result.raw
        try:
            output = parse_output(result, model)
        except ValueError as e2:
            error = str(e2)[:500]

    write_json(path, {"output": output.model_dump() if output else None, "raw": raw, "error": error,
                      "model": PRIMARY_MODEL, "prompt_version": PROMPT_VERSION,
                      "saved_at": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                      "requests": requests_today() - before})
    return output, requests_today() - before

In [ ]:
# ✅ Checkpoint 6, part 1: offline tests (no API calls)
assert parse_json_text('```json\n{"a": 1}\n```', type("M", (BaseModel,), {"__annotations__": {"a": int}})).a == 1
assert parse_json_text('Sure! Here it is: {"a": 2} Hope that helps.', type("M", (BaseModel,), {"__annotations__": {"a": int}})).a == 2
for bad in ["no json here", '{"a": "not a number"}']:
    try:
        parse_json_text(bad, type("M", (BaseModel,), {"__annotations__": {"a": int}}))
        raise AssertionError("should have failed: " + bad)
    except ValueError:
        pass

assert error_kind(Exception("429 RESOURCE_EXHAUSTED quotaId: GenerateRequestsPerDayPerProjectPerModel-FreeTier")) == "daily"
assert error_kind(Exception("429 RESOURCE_EXHAUSTED quotaId: GenerateRequestsPerMinutePerProjectPerModel")) == "retry"
assert error_kind(Exception("503 UNAVAILABLE: the model is overloaded")) == "retry"
assert error_kind(ValueError("bad input")) is None
print("offline plumbing tests passed")

In [ ]:
# ✅ Checkpoint 6, part 2: one real cached step, run twice. The first run makes 1 request
# (only the very first time this notebook runs; after that it's cached too); the second makes 0.
def build_smoke(note):
    agent = make_agent("Content Analyst",
                       "Identify what an email is trying to make the reader do, and which manipulation tactics it uses.",
                       "You study phishing emails and describe them precisely, quoting the email as evidence.")
    return single_task_crew(agent, "Analyse this email. Quote its exact words as evidence for each tactic."
                                   "\n\nEMAIL:\n{email}" + note, ContentReport)

for run in (1, 2):
    report, new_requests = run_step("checkpoint", "smoke_email", "content", build_smoke, {"email": SMOKE_EMAIL}, ContentReport)
    print(f"run {run}: new requests = {new_requests}, intent = {report.intent if report else 'FAILED'}")
print("requests counted today (this storage location):", requests_today())

**✅ Checkpoint 6:** `offline plumbing tests passed`; run 1 shows 1 new request (or 0 if it was cached
on an earlier run) and run 2 shows **0**, with the same intent both times.

## Stage 7: The agents and the disagreement round

**How one email flows through the full system** (at most 4 LLM requests, plus a repair if an answer can't be parsed):

1. **Technical Analyst:** the Stage 4 tools, 0 requests.
2. **Content Analyst:** 1 request. It sees the subject, the body text and the list of link *domains*, but no headers.
   Python then checks that every quote it gives really appears in the email, and drops any that don't (counted as hallucinations).
3. **Python decides whether the analysts disagree:** the scores are 40+ points apart, or one says 70+ while the other says under 40.
4. **Verdict agent, draft:** 1 request. It merges both reports. If Python flagged a disagreement, it must also write
   **one** question to **one** analyst: the one whose conclusion most needs checking.
5. **Only if there is a disagreement:**
   - **Follow-up:** 1 request to the analyst being asked. The Technical Analyst's LLM gets its findings plus the key header values
     and every link URL, so it can look beyond the fixed checks. The Content Analyst gets the email and its earlier report.
   - **Verdict agent, final:** 1 request, with the question and answer added.

**How the cap works.** The follow-up is a single `if` in Python, not a loop, so there is no code path to a second round.
Agents can't delegate to each other (`allow_delegation=False`) and each step is its own one-task crew.
CrewAI's own hierarchical or delegation modes were not used: they don't bound the number of calls and are harder to explain.

**Score and label.** The Verdict agent gives the risk score; Python turns it into the label with the same thresholds as everything else,
so the label can never contradict the score.

The Technical Analyst's follow-up gets header values and URLs written into its prompt rather than calling tools itself:
Flash Lite models are unreliable at tool calling, and each tool call would cost an extra request.

In [ ]:
from typing import Optional

class EvidenceItem(BaseModel):
    id: int
    source: Literal["technical", "content", "followup"]
    finding: str
    detail: str = Field(description="the exact header value, URL or quote")
    severity: Literal["low", "medium", "high"]

class Reason(BaseModel):
    text: str = Field(description="one plain-English sentence for a non-expert")
    evidence_ids: list[int]

class FollowupQuestion(BaseModel):
    target: Literal["technical", "content"]
    question: str

class VerdictBody(BaseModel):
    risk_score: int = Field(ge=0, le=100)
    confidence: Literal["low", "medium", "high"]
    evidence: list[EvidenceItem]
    reasons: list[Reason]
    recommended_action: str

class VerdictDraft(VerdictBody):
    followup: Optional[FollowupQuestion] = None

class FollowupAnswer(BaseModel):
    answer: str
    revised_score: Optional[int] = Field(default=None, ge=0, le=100, description="your updated risk, or null if unchanged")
    evidence: list[str]

class Disagreement(BaseModel):
    tech_score: int
    content_risk: int
    target: str
    question: str
    answer: str
    score_before: int    # the Verdict agent's draft score
    score_after: int     # its final score

class Verdict(BaseModel):
    """What the system returns for one email."""
    email_id: str
    verdict: Literal["phishing", "suspicious", "legitimate", "error"]
    risk_score: int
    confidence: str
    evidence: list[EvidenceItem]
    reasons: list[Reason]
    recommended_action: str
    disagreement: Optional[Disagreement] = None
    meta: dict

In [ ]:
def collapse(text, limit=BODY_CHAR_LIMIT):
    return re.sub(r"\s+", " ", text).strip()[:limit]

def header_summary(raw):
    """The header values the Technical Analyst's LLM may look at. Date and routing headers are left out (era gap)."""
    msg = email.message_from_string(raw, policy=policy.default)
    lines = []
    for name in ("From", "Reply-To", "Return-Path", "Subject", "List-Id", "Authentication-Results"):
        try:
            value = msg.get(name)
        except Exception:
            value = None
        if value:
            lines.append(f"{name}: {collapse(str(value), 300)}")
    return "\n".join(lines)

def tech_summary(tech):
    lines = [f"- [{f.severity}] {f.check}: {f.detail} Evidence: {f.evidence}" for f in tech.findings]
    if tech.auth_available:
        auth = (", ".join(f"{k}={v}" for k, v in tech.auth.items()) +
                ". A pass only proves which domain sent the email, not that the email is safe.")
    else:
        auth = ("no authentication headers. That is normal for older or internally relayed mail and is "
                "NOT evidence either way.")
    return ("\n".join(lines) or "- No technical red flags found.") + f"\n(Authentication: {auth})"

def check_quotes(report, text):
    """Drop tactics whose quote doesn't appear in the email. Returns (cleaned report, number dropped)."""
    def norm(s):
        s = s.replace("\u2019", "'").replace("\u2018", "'").replace("\u201c", '"').replace("\u201d", '"')
        return re.sub(r"\s+", " ", s).strip().lower()
    haystack = norm(text)
    kept = [t for t in report.tactics if norm(t.quote) and norm(t.quote) in haystack]
    return report.model_copy(update={"tactics": kept}), len(report.tactics) - len(kept)

def analysts_disagree(tech_score, content_risk):
    hi, lo = max(tech_score, content_risk), min(tech_score, content_risk)
    return hi - lo >= 40 or (hi >= PHISHING_THRESHOLD and lo < SUSPICIOUS_THRESHOLD)

### The prompts

Each prompt is a CrewAI task description. Words in `{braces}` are filled in from the email at run time.
If you change a prompt, bump `PROMPT_VERSION` in the config cell so old cached answers aren't reused.

In [ ]:
CONTENT_TASK = """Analyse the email below for phishing intent and manipulation tactics. Judge the wording only.

- intent: what the email wants the reader to do. Use "benign" for an ordinary email that asks for nothing risky.
- tactics: each manipulation tactic used (urgency, authority, fear, reward, scarcity, curiosity), each with a short
  quote of at most 20 words copied EXACTLY from the email. Use an empty list if there are none.
- content_risk: 0-100, how likely the wording alone is phishing. Ordinary discussions, newsletters and notifications
  that don't push the reader to log in, pay, or hand over information score low.
- summary: one or two sentences.

SUBJECT: {subject}
LINK DOMAINS (context only): {domains}
BODY:
{body}"""

VERDICT_TASK = """You combine two analysts' reports about one email into a verdict for an ordinary reader.

TECHNICAL REPORT (automatic checks of headers and links; technical score {tech_score}/100):
{tech}

CONTENT REPORT (a reading of the wording; content risk {content_risk}/100):
{content}
{followup_section}
How to weigh them: technical red flags raise the risk, but clean technical evidence does NOT clear an email.
Many scams (gift-card requests, fake invoices, CEO fraud) come from ordinary accounts with nothing technically wrong;
a pressured request for money, credentials, codes or secrecy is phishing on its own.

Write:
- risk_score: 0-100. 70 or more means phishing, 40-69 suspicious, under 40 legitimate.
- confidence: low, medium or high.
- evidence: the specific findings that matter, numbered from 1, each copied from the reports with its exact detail
  (header value, URL or quote), its source (technical, content or followup) and a severity. Absent signals,
  such as missing authentication headers, are not evidence.
- reasons: two to four plain-English sentences a non-expert can follow, each citing the evidence ids it rests on.
- recommended_action: one or two sentences telling the reader what to do.
{instruction}"""

TECH_FOLLOWUP_TASK = """The verdict agent has a question about the technical evidence for an email.

AUTOMATIC CHECK RESULTS (technical score {tech_score}/100):
{tech}

HEADER VALUES:
{headers}

ALL LINK URLS:
{urls}

QUESTION: {question}

Answer using only the technical evidence above. Look beyond the automatic checks: for example, whether the sending
domain plausibly belongs to the organisation the email claims to be from. Missing authentication headers are
not evidence of anything; only an explicit fail is. If the technical evidence is simply clean, say so: clean
headers show nothing is technically wrong, not that the email is safe. Give revised_score as your updated
technical risk (0-100), or null if your view is unchanged, and list the specific evidence you relied on."""

CONTENT_FOLLOWUP_TASK = """The verdict agent has a question about your reading of an email.

YOUR EARLIER REPORT:
{content}

SUBJECT: {subject}
BODY:
{body}

QUESTION: {question}

Answer from the wording of the email. Give revised_score as your updated content risk (0-100), or null if your view
is unchanged, and list the exact quotes you relied on."""

def content_agent():
    return make_agent("Content Analyst",
        "Identify what an email wants its reader to do and which manipulation tactics it uses.",
        "You study phishing and social engineering. You quote the email exactly and don't exaggerate: "
        "most email is harmless.")

def technical_agent():
    return make_agent("Technical Analyst",
        "Judge an email's headers and links.",
        "You are an email-security engineer. You reason from concrete technical evidence: domains, headers, URLs.")

def verdict_agent():
    return make_agent("Verdict and Explainer",
        "Turn two analysts' reports into a fair verdict with clear, evidence-backed reasons.",
        "You explain email risks to non-experts. You weigh evidence, you don't just add it up, and every reason you "
        "give points at specific evidence.")

In [ ]:
def analyze_email(email_id, raw, config="full"):
    """The full system for one email. Returns a Verdict; every LLM step is cached."""
    p = parse_email(raw)
    tech = analyze_technical(raw)
    body = collapse(p.body_text)
    steps, requests, failed = [], 0, []

    def step(name, build, inputs, model):
        nonlocal requests
        out, n = run_step(config, email_id, name, build, inputs, model)
        steps.append(name)
        requests += n
        if out is None:
            failed.append(name)
        return out

    # 1. Content Analyst
    content = step("content", lambda note: single_task_crew(content_agent(), CONTENT_TASK + note, ContentReport),
                   {"subject": collapse(p.subject, 300), "domains": ", ".join(tech.url_domains[:20]) or "(none)",
                    "body": body}, ContentReport)
    hallucinated = 0
    if content:
        content, hallucinated = check_quotes(content, p.subject + " " + body)
    content_risk = content.content_risk if content else tech.tech_score   # if it failed, don't invent a disagreement
    content_text = content.model_dump_json(indent=1) if content else "(the content analysis failed)"

    # 2. Do the analysts disagree? (decided by Python, not by an LLM)
    disagree = analysts_disagree(tech.tech_score, content_risk)
    instruction = (f"The analysts DISAGREE (technical {tech.tech_score} vs content {content_risk}). Your risk_score is "
                   "provisional. Fill followup with ONE specific question to the analyst whose conclusion most needs "
                   "checking (target technical or content)." if disagree else "The analysts agree. Set followup to null.")
    verdict_inputs = {"tech_score": tech.tech_score, "tech": tech_summary(tech), "content_risk": content_risk,
                      "content": content_text, "followup_section": "", "instruction": instruction}

    # 3. Verdict draft
    draft = step("verdict_draft", lambda note: single_task_crew(verdict_agent(), VERDICT_TASK + note, VerdictDraft),
                 verdict_inputs, VerdictDraft)
    final, disagreement = draft, None

    # 4. The disagreement round: a single if, so at most one follow-up ever happens.
    if disagree and draft is not None:
        if draft.followup:
            target, question = draft.followup.target, draft.followup.question
        else:   # the draft forgot the question: ask whichever analyst gave the lower score to re-check
            target = "technical" if tech.tech_score < content_risk else "content"
            question = "The other analyst reached a very different conclusion. Re-check your evidence: is this email phishing?"
        if target == "technical":
            answer = step("followup", lambda note: single_task_crew(technical_agent(), TECH_FOLLOWUP_TASK + note, FollowupAnswer),
                          {"tech_score": tech.tech_score, "tech": tech_summary(tech), "headers": header_summary(raw),
                           "urls": "\n".join(p.urls[:30]) or "(none)", "question": question}, FollowupAnswer)
        else:
            answer = step("followup", lambda note: single_task_crew(content_agent(), CONTENT_FOLLOWUP_TASK + note, FollowupAnswer),
                          {"content": content_text, "subject": collapse(p.subject, 300), "body": body,
                           "question": question}, FollowupAnswer)
        if answer is not None:
            followup_section = (f"\nFOLLOW-UP: you asked the {target} analyst: {question}\n"
                                f"Answer: {answer.answer}\nRevised score: {answer.revised_score}\n"
                                f"Evidence: {'; '.join(answer.evidence)}\n")
            final = step("verdict_final", lambda note: single_task_crew(verdict_agent(), VERDICT_TASK + note, VerdictBody),
                         {**verdict_inputs, "followup_section": followup_section,
                          "instruction": "Give your final verdict, taking the follow-up answer into account."},
                         VerdictBody) or draft
            disagreement = Disagreement(tech_score=tech.tech_score, content_risk=content_risk, target=target,
                                        question=question, answer=answer.answer,
                                        score_before=draft.risk_score, score_after=final.risk_score)

    meta = {"config": config, "model": PRIMARY_MODEL, "prompt_version": PROMPT_VERSION, "steps": steps,
            "requests": requests, "failed_steps": failed, "hallucinated_quotes": hallucinated,
            "tech_score": tech.tech_score, "content_risk": content.content_risk if content else None}
    if final is None:
        return Verdict(email_id=email_id, verdict="error", risk_score=tech.tech_score, confidence="low",
                       evidence=[], reasons=[], recommended_action="The analysis failed; treat this email with care.",
                       meta=meta)
    ids = {e.id for e in final.evidence}
    reasons = [r.model_copy(update={"evidence_ids": [i for i in r.evidence_ids if i in ids]}) for r in final.reasons]
    verdict = Verdict(email_id=email_id, verdict=score_to_label(final.risk_score), risk_score=final.risk_score,
                      confidence=final.confidence, evidence=final.evidence, reasons=reasons,
                      recommended_action=final.recommended_action, disagreement=disagreement, meta=meta)
    write_json(f"{DRIVE_ROOT}/results/{config}/{email_id}.json", verdict.model_dump())
    return verdict

In [ ]:
def print_verdict(v, title=""):
    print(f"\n{'=' * 80}\n{title}  [{v.email_id}]")
    print(f"VERDICT: {v.verdict.upper()}   risk {v.risk_score}/100   confidence {v.confidence}")
    print(f"(technical score {v.meta['tech_score']}, content risk {v.meta['content_risk']}, "
          f"steps: {', '.join(v.meta['steps'])}; {v.meta['requests']} new requests)")
    print("\nReasons:")
    for r in v.reasons:
        print(f"  - {r.text}  [evidence {', '.join(map(str, r.evidence_ids))}]")
    print("Evidence:")
    for e in v.evidence:
        print(f"  {e.id}. ({e.source}, {e.severity}) {e.finding}: {e.detail[:150]}")
    if v.disagreement:
        d = v.disagreement
        print(f"Disagreement round: technical {d.tech_score} vs content {d.content_risk}")
        print(f"  asked the {d.target} analyst: {d.question}")
        print(f"  answer: {d.answer[:400]}")
        print(f"  verdict score {d.score_before} -> {d.score_after}")
    print(f"Recommended action: {v.recommended_action}")

In [ ]:
# ✅ Checkpoint 7: the 3 demo emails, plus one email made up to make the analysts clash
# (clean headers, manipulative body), which must trigger exactly one follow-up.
CLASH_EMAIL = """From: Laura Bennett <laura.bennett@example.com>
To: reader@example.com
Subject: Quick favour - need this done in the next hour

Hi,

I'm stuck in back-to-back board meetings and can't talk on the phone. I need you to buy five $200 Apple gift cards
for a client appreciation surprise. It has to be done within the hour, and please keep it confidential until it's
announced. Scratch off the backs and reply to this email with the codes. I'll make sure you're reimbursed today.

Thanks,
Laura
CEO"""

demo_verdicts = {}
for role, eid in splits["demo"].items():
    demo_verdicts[role] = analyze_email(eid, emails.at[eid, "raw"])
    print_verdict(demo_verdicts[role], role)

clash = analyze_email("clash_test", CLASH_EMAIL, config="checkpoint")
print_verdict(clash, "made-up clash case")

for v in [*demo_verdicts.values(), clash]:
    assert len(v.meta["steps"]) <= 4, "more than 4 LLM steps for one email"
assert clash.disagreement is not None and clash.meta["steps"].count("followup") == 1
print("\nchecks passed; requests counted today:", requests_today())

**✅ Checkpoint 7:** each of the four emails prints a verdict, reasons that cite evidence, and a recommended action.
No email takes more than 4 steps. The made-up clash case shows exactly one follow-up round.

## Stage 8: The LLM-only configuration

The ablation the full system is compared against: one prompt, one request, no tools and no agents talking to each other.
It sees what the full system's agents can see between them (subject, sender headers, link URLs, body), minus
the Authentication-Results header, which only the deterministic checks read. Date and routing headers are left out,
as everywhere else (era gap).

In [ ]:
LLM_ONLY_TASK = """Decide whether the email below is phishing, and explain your verdict to an ordinary reader.

HEADERS:
{headers}

LINK URLS:
{urls}

SUBJECT: {subject}
BODY:
{body}

Judge both the technical details (sender domain, reply-to address, where the links really go) and the wording
(what the email asks the reader to do, and how it pressures them). Clean-looking technical details do NOT clear
an email: a pressured request for money, credentials, codes or secrecy is phishing on its own.

Write:
- risk_score: 0-100. 70 or more means phishing, 40-69 suspicious, under 40 legitimate.
- confidence: low, medium or high.
- evidence: the specific findings that matter, numbered from 1, each with its exact detail (header value, URL or
  quote copied from the email), its source (technical for headers and links, content for the wording) and a severity.
- reasons: two to four plain-English sentences a non-expert can follow, each citing the evidence ids it rests on.
- recommended_action: one or two sentences telling the reader what to do."""

def llm_only_agent():
    return make_agent("Email Security Analyst",
        "Decide whether an email is phishing and explain why, citing specific evidence.",
        "You check emails for phishing. You reason from concrete evidence (domains, links, the exact wording) and "
        "don't exaggerate: most email is harmless.")

def analyze_llm_only(email_id, raw, config="llm_only"):
    """The LLM-only configuration for one email: a single cached step. Returns a Verdict."""
    p = parse_email(raw)
    headers = "\n".join(l for l in header_summary(raw).splitlines() if not l.startswith("Authentication-Results"))
    inputs = {"headers": headers or "(none)", "urls": "\n".join(p.urls[:30]) or "(none)",
              "subject": collapse(p.subject, 300), "body": collapse(p.body_text)}
    out, requests = run_step(config, email_id, "llm_only",
                             lambda note: single_task_crew(llm_only_agent(), LLM_ONLY_TASK + note, VerdictBody),
                             inputs, VerdictBody)
    meta = {"config": config, "model": PRIMARY_MODEL, "prompt_version": PROMPT_VERSION, "steps": ["llm_only"],
            "requests": requests, "failed_steps": [] if out else ["llm_only"], "tech_score": None, "content_risk": None}
    if out is None:
        return Verdict(email_id=email_id, verdict="error", risk_score=0, confidence="low", evidence=[], reasons=[],
                       recommended_action="The analysis failed; treat this email with care.", meta=meta)
    ids = {e.id for e in out.evidence}
    reasons = [r.model_copy(update={"evidence_ids": [i for i in r.evidence_ids if i in ids]}) for r in out.reasons]
    verdict = Verdict(email_id=email_id, verdict=score_to_label(out.risk_score), risk_score=out.risk_score,
                      confidence=out.confidence, evidence=out.evidence, reasons=reasons,
                      recommended_action=out.recommended_action, meta=meta)
    write_json(f"{DRIVE_ROOT}/results/{config}/{email_id}.json", verdict.model_dump())
    return verdict

In [ ]:
# ✅ Checkpoint 8: one email through the LLM-only configuration (1 request, 0 on a rerun)
eid = splits["demo"]["subtle_phish"]
v = analyze_llm_only(eid, emails.at[eid, "raw"])
print_verdict(v, "LLM-only, subtle demo email")
assert v.verdict != "error" and v.meta["steps"] == ["llm_only"]
print("\nchecks passed; requests counted today:", requests_today())

**✅ Checkpoint 8:** the email prints a parsed verdict with reasons and evidence from a single step.
Running the cell again makes 0 new requests (cached).

## Stage 9: Evaluation

Both LLM configurations run over the 100-email subset. Every step is cached, so this cell is resumable: if it stops
(daily budget reached, or the API stays overloaded), run it again later and it continues where it left off.
Budget: about 100 requests for LLM-only and 2-4 per email for the full system, so it likely needs two quota days.

In [ ]:
RUNNERS = {"llm_only": analyze_llm_only, "full": analyze_email}

def run_evaluation():
    """Analyse every subset email with each LLM configuration. Returns {config: {email_id: Verdict}}, possibly partial."""
    done = {config: {} for config in RUNNERS}
    for config, analyze in RUNNERS.items():
        for n, eid in enumerate(subset_ids, 1):
            try:
                v = analyze(eid, emails.at[eid, "raw"])
            except DailyBudgetReached as e:
                print(f"STOPPED: {e}")
                return done
            except Exception as e:   # e.g. still overloaded after all retries; nothing is cached, so a rerun tries again
                print(f"   {config} {eid}: {type(e).__name__}: {str(e)[:200]} (skipped for now)")
                continue
            done[config][eid] = v
            if v.meta["requests"]:
                print(f"{config} {n}/{len(subset_ids)} {eid}: {v.verdict} ({v.meta['requests']} requests, "
                      f"{requests_today()} today)")
    return done

eval_verdicts = run_evaluation()
for config, got in eval_verdicts.items():
    print(f"{config}: {len(got)}/{len(subset_ids)} emails done")

In [ ]:
# ✅ Checkpoint 9: the results table and confusion matrices
complete = all(len(got) == len(subset_ids) for got in eval_verdicts.values())
if not complete:
    print("WARNING: the evaluation is not finished; LLM rows cover only the emails done so far. Rerun the cell above.\n")

rows, tables = {}, {}
def add_row(name, ids, verdicts):
    truth = [emails.at[eid, "label"] for eid in ids]
    rows[name] = {**evaluate(truth, verdicts), "emails": len(ids), "errors": verdicts.count("error"), "requests": 0}
    tables[name] = confusion_table(truth, verdicts)

add_row("tools only", subset_ids, tools_verdicts)
add_row("TF-IDF baseline", subset_ids, baseline_verdicts)
for config, name in [("llm_only", "LLM only"), ("full", "full system")]:
    ids = [eid for eid in subset_ids if eid in eval_verdicts[config]]
    verdicts = [eval_verdicts[config][eid].verdict for eid in ids]
    add_row(name, ids, verdicts)

# Requests per email, read from the cache (the verdicts' own counts are 0 for cached steps)
def cached_requests(config, eid):
    folder = f"{CACHE_DIR}/{PRIMARY_MODEL}/{PROMPT_VERSION}/{config}/{eid}"
    total = 0
    for name in os.listdir(folder) if os.path.isdir(folder) else []:
        with open(f"{folder}/{name}", encoding="utf-8") as f:
            total += json.load(f)["requests"]
    return total

for config, name in [("llm_only", "LLM only"), ("full", "full system")]:
    rows[name]["requests"] = sum(cached_requests(config, eid) for eid in eval_verdicts[config])

results = pd.DataFrame(rows).T
results[["emails", "errors", "requests"]] = results[["emails", "errors", "requests"]].astype(int)
print(results.to_string(), "\n")
for name, table in tables.items():
    print(f"--- {name}\n{table}\n")

full = list(eval_verdicts["full"].values())
print("full system: disagreement rounds", sum(v.disagreement is not None for v in full),
      "| verdicts changed by the round", sum(v.disagreement is not None and
          score_to_label(v.disagreement.score_before) != score_to_label(v.disagreement.score_after) for v in full),
      "| quotes dropped as not in the email", sum(v.meta.get("hallucinated_quotes", 0) for v in full))
write_json(f"{DRIVE_ROOT}/results/summary.json", {"complete": complete, "model": PRIMARY_MODEL,
                                                  "prompt_version": PROMPT_VERSION, "rows": rows})

**✅ Checkpoint 9:** a table with precision, recall and F1 for tools only, the TF-IDF baseline, LLM only and the
full system, all on the same 100 emails, plus a confusion matrix for each. Failed analyses show up in the
"errors" column and count as wrong predictions. No warning above the table means the evaluation is complete.